# Local NER Model Test — Qwen variants

Tests multiple local Qwen models for NER extraction against cached Gemini results.
Runs the same prompt + page text through each model and compares entry counts,
firm name overlap, city fill rate, and speed.

**Before you start:**
- Enable a GPU runtime: Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)
- Upload your `ner_prompt.md` and a handful of `*_aligned.json` files in the next cell
- Optionally: upload matching `*_entries.json` files for Gemini comparison

Models tested (edit the `MODELS` list in the config cell to add/remove):
- `Qwen/Qwen3.5-9B` — 9.7B thinking model, 4-bit
- `Qwen/Qwen3.5-4B` — 4B thinking model, 4-bit (faster, less accurate)
- `Qwen/Qwen2.5-7B-Instruct` — 7B non-thinking, 4-bit (no thinking overhead)

---
## 1. Setup

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU runtime first')

In [ ]:
%pip install -q transformers>=4.51 bitsandbytes>=0.43 accelerate>=0.27 huggingface_hub

---
## 2. Upload test files

Upload:
- `ner_prompt.md` — the NER system prompt
- One or more `*_gemini-2.0-flash_aligned.json` files (page text)
- Optionally: matching `*_gemini-3.1-flash-lite-preview_entries.json` files (for comparison)

Tip: grab 5–10 pages that include a mix of front matter and dense directory pages.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # select files in the dialog

WORK_DIR = Path('/content/ner_test')
WORK_DIR.mkdir(exist_ok=True)

for fname, data in uploaded.items():
    (WORK_DIR / fname).write_bytes(data)
    print(f'  saved: {fname} ({len(data):,} bytes)')

aligned_files = sorted(WORK_DIR.glob('*_aligned.json'))
prompt_file   = next(WORK_DIR.glob('ner_prompt.md'), None)
print(f'\nPrompt: {prompt_file}')
print(f'Aligned pages: {len(aligned_files)}')
for f in aligned_files:
    print(f'  {f.name}')

---
## 3. Configuration — edit here

In [ ]:
# ── Models to test ────────────────────────────────────────────────────────────
# Each entry: (display_name, hf_model_id)
MODELS = [
    ('Qwen3.5-9B (4-bit)',       'Qwen/Qwen3.5-9B'),
    ('Qwen3.5-4B (4-bit)',       'Qwen/Qwen3.5-4B'),
    ('Qwen2.5-7B-Instruct (4-bit)', 'Qwen/Qwen2.5-7B-Instruct'),
]

# ── Gemini slug for loading cached comparison entries ─────────────────────────
GEMINI_NER_SLUG = 'gemini-3.1-flash-lite-preview'

# ── Aligned model slug used in *_aligned.json filenames ───────────────────────
ALIGNED_MODEL   = 'gemini-2.0-flash'

# ── Max tokens to generate per page ──────────────────────────────────────────
MAX_NEW_TOKENS  = 4096

---
## 4. Helper functions

In [ ]:
import json, re, time

def _normalize_line(text):
    text = text.strip()
    text = re.sub(r'[.\s]{3,}', '\t', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def page_text_from_aligned(aligned):
    lines = [
        _normalize_line(ln.get('gemini_text', ''))
        for ln in aligned.get('lines', [])
        if ln.get('gemini_text', '').strip()
    ]
    lines += [_normalize_line(t) for t in aligned.get('unmatched_gemini', []) if t.strip()]
    return '\n'.join(lines)

def build_user_message(page_text, prior_context):
    ctx = '\n'.join(f'{k}: {v}' for k, v in prior_context.items()) or '(none)'
    return f'## Prior page context\n{ctx}\n\n## Page text\n{page_text}\n\nReturn the JSON extraction.'

def strip_fence(text):
    text = text.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        inner = lines[1:]
        if inner and inner[-1].strip() == '```':
            inner = inner[:-1]
        text = '\n'.join(inner).strip()
    return text

def parse_json(text):
    text = strip_fence(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find('{'), text.rfind('}') + 1
        if 0 <= start < end:
            try:
                return json.loads(text[start:end])
            except json.JSONDecodeError:
                pass
    return None

def field_fill(entries, field):
    if not entries:
        return 0.0
    return sum(1 for e in entries if e.get(field) and str(e[field]).strip()) / len(entries)

def compare(gemini_entries, local_entries):
    g = {e.get('firm_name') or e.get('business_name') or '' for e in gemini_entries}
    l = {e.get('firm_name') or e.get('business_name') or '' for e in local_entries}
    g.discard(''); l.discard('')
    return {
        'gem': len(gemini_entries), 'loc': len(local_entries),
        'overlap': len(g & l),
        'only_g': len(g - l), 'only_l': len(l - g),
        'city_g': field_fill(gemini_entries, 'city'),
        'city_l': field_fill(local_entries, 'city'),
    }

print('Helpers loaded.')

---
## 5. Inference function (transformers + 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_loaded = {}  # cache: model_id -> (model, tokenizer)

def load_model(model_id):
    if model_id in _loaded:
        return _loaded[model_id]
    print(f'Loading {model_id} in 4-bit…')
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )
    mdl.eval()
    _loaded[model_id] = (mdl, tok)
    print(f'  Loaded. Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')
    return mdl, tok


def call_model(model_id, system_prompt, user_text):
    """Returns (response_text, elapsed_seconds)."""
    mdl, tok = load_model(model_id)

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_text},
    ]

    # apply_chat_template with enable_thinking=False suppresses Qwen3 thinking mode
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        # Older tokenizer / non-thinking model — no enable_thinking param
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = tok(text, return_tensors='pt').to(mdl.device)
    input_len = inputs.input_ids.shape[1]

    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,        # greedy = deterministic
            pad_token_id=tok.eos_token_id,
        )
    elapsed = time.time() - t0

    response = tok.decode(out[0][input_len:], skip_special_tokens=True)
    return response, elapsed


print('Inference function ready.')

---
## 6. Run the test

Runs all models across all uploaded pages and prints a comparison table.
Results are stored in `all_results` for further analysis.

In [ ]:
from pathlib import Path

if not prompt_file:
    raise FileNotFoundError('ner_prompt.md not found — re-run the upload cell')
if not aligned_files:
    raise FileNotFoundError('No *_aligned.json files found — re-run the upload cell')

system_prompt = prompt_file.read_text(encoding='utf-8')
aligned_suffix = f'_{ALIGNED_MODEL}_aligned.json'
gemini_slug = GEMINI_NER_SLUG.replace('/', '_').replace(':', '_')

# all_results: {model_display_name: {page_stem: {cmp, elapsed, ok, entries}}}
all_results = {}

COL = 28  # column width for page stem

for display_name, model_id in MODELS:
    print(f'\n{"+"*70}')
    print(f'  {display_name}  ({model_id})')
    print(f'{"+"*70}')
    print(f'{"Page":<{COL}} {"Gem":>5} {"Loc":>5} {"Ovlp%":>6} {"CitL%":>6} {"s":>6} {"JSON?":>6}')
    print('-' * 65)

    context = {}
    page_results = {}
    total_time = 0.0
    json_fails = 0

    for af in aligned_files:
        stem = af.name[: -len(aligned_suffix)] if af.name.endswith(aligned_suffix) else af.stem

        aligned   = json.loads(af.read_text(encoding='utf-8'))
        page_text = page_text_from_aligned(aligned)
        user_msg  = build_user_message(page_text, context)

        # Load cached Gemini entries for comparison (optional)
        gemini_cache = WORK_DIR / f'{stem}_{gemini_slug}_entries.json'
        gemini_entries = []
        if gemini_cache.exists():
            try:
                gemini_entries = json.loads(gemini_cache.read_text())['entries']
            except Exception:
                pass

        raw, elapsed = call_model(model_id, system_prompt, user_msg)
        total_time += elapsed
        parsed = parse_json(raw)
        ok = parsed is not None

        if not ok:
            json_fails += 1
            local_entries = []
            context = context
        else:
            local_entries = parsed.get('entries', [])
            context = parsed.get('page_context', context)

        cmp = compare(gemini_entries, local_entries)
        page_results[stem] = {'cmp': cmp, 'elapsed': elapsed, 'ok': ok, 'entries': local_entries}

        ovlp_pct = cmp['overlap'] * 100 // cmp['gem'] if cmp['gem'] else 0
        print(
            f'{stem:<{COL}} '
            f'{cmp["gem"]:>5} '
            f'{cmp["loc"]:>5} '
            f'{ovlp_pct:>5}% '
            f'{cmp["city_l"]*100:>5.0f}% '
            f'{elapsed:>5.1f}s '
            f'{"✓" if ok else "✗ FAIL":>6}'
        )

    all_results[display_name] = page_results

    n = len(aligned_files)
    avg = total_time / n if n else 0
    print(f'\nTotal: {total_time:.0f}s  avg: {avg:.1f}s/page  JSON failures: {json_fails}')

print('\n\n=== All models done ===')

---
## 7. Summary comparison table

In [ ]:
print(f'\n{"Model":<35} {"Pages":>6} {"AvgOvlp%":>9} {"AvgCit%":>8} {"AvgSec":>7} {"Fails":>6}')
print('-' * 75)

for display_name, page_results in all_results.items():
    pages = [v for v in page_results.values() if v['cmp']['gem'] > 0]  # only pages with Gemini data
    all_pages = list(page_results.values())

    if pages:
        avg_ovlp = sum(
            v['cmp']['overlap'] / v['cmp']['gem'] for v in pages
        ) / len(pages) * 100
    else:
        avg_ovlp = 0.0

    dir_pages = [v for v in all_pages if v['cmp']['loc'] > 0]
    avg_city = sum(v['cmp']['city_l'] for v in dir_pages) / len(dir_pages) * 100 if dir_pages else 0
    avg_sec  = sum(v['elapsed'] for v in all_pages) / len(all_pages) if all_pages else 0
    fails    = sum(1 for v in all_pages if not v['ok'])

    print(f'{display_name:<35} {len(all_pages):>6} {avg_ovlp:>8.0f}% {avg_city:>7.0f}% {avg_sec:>6.0f}s {fails:>6}')

print()
print('Note: AvgOvlp% only counts pages where Gemini comparison data was uploaded.')
print('      AvgSec includes fast front-matter pages — directory pages will be slower.')

---
## 8. Export entries from best model

In [ ]:
# Change this to whichever model performed best
BEST_MODEL = MODELS[0][0]  # first model by default

entries_out = []
for page_data in all_results[BEST_MODEL].values():
    entries_out.extend(page_data['entries'])

out_path = WORK_DIR / 'local_ner_output.json'
out_path.write_text(json.dumps(entries_out, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Wrote {len(entries_out)} entries to {out_path}')

# Download the file
from google.colab import files
files.download(str(out_path))